In [1]:
import sys
sys.path.append('../')

# NaHCO3-CO2(g) equilibrium
- Calculate pH of 0.01 mol/L NaHCO3 aq. before and after contact with 400 ppm CO2(g).
- Considers activity coefficient (Davies model)

In [2]:
from pprint import pprint
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

## Settings

In [3]:
import eq_solver

### System and Conditions

In [4]:
c = 0.01
s_gas = eq_solver.System.from_yaml('./Na-CO3-gas.yaml')
s_nogas = eq_solver.System.from_yaml('./Na-CO3-nogas.yaml')
c_gas = eq_solver.Conditions.from_dict(s_gas, {'sodium': c, 'carbonate': 400e-6})
c_nogas = eq_solver.Conditions.from_dict(s_nogas, {'sodium': c, 'carbonate': c})

In [5]:
# structure of System
pprint(s_gas)
display(pd.concat([
    s_gas.stoichiometry_matrix_as_df(),
    s_gas.logK_as_df()
    ], axis='columns'))


System(activity_model='davies',
       species=(Species(name='H+'),
                Species(name='OH-'),
                Species(name='Na+'),
                Species(name='CO2aq'),
                Species(name='HCO3-'),
                Species(name='CO3^2-'),
                Species(name='CO2(g)')),
       components=(Component(name='proton',
                             base_species=Species(name='H+'),
                             constraint=<Constraint.CHARGE: 1>,
                             charge=1),
                   Component(name='carbonate',
                             base_species=Species(name='CO2(g)'),
                             constraint=<Constraint.DIRECT: 2>,
                             charge=0),
                   Component(name='sodium',
                             base_species=Species(name='Na+'),
                             constraint=<Constraint.DIRECT: 2>,
                             charge=1)),
       temperature=298.15,
       specs=SystemSpecs(n_vars=2

,H+,CO2(g),Na+,logK
H+,1.0,-0.0,-0.0,0.00
OH-,-1.0,0.0,0.0,-14.00
Na+,0.0,0.0,1.0,0.00
CO2aq,0.0,1.0,0.0,-1.47
HCO3-,-1.0,1.0,0.0,-7.77
CO3^2-,-2.0,1.0,0.0,-18.10
CO2(g),0.0,1.0,0.0,-0.00


In [6]:
pprint(c_nogas)

Conditions(values=array([0.  , 0.01, 0.01]),
           system=System(activity_model='davies',
                         species=(Species(name='H+'),
                                  Species(name='OH-'),
                                  Species(name='Na+'),
                                  Species(name='CO2aq'),
                                  Species(name='HCO3-'),
                                  Species(name='CO3^2-')),
                         components=(Component(name='proton',
                                               base_species=Species(name='H+'),
                                               constraint=<Constraint.CHARGE: 1>,
                                               charge=1),
                                     Component(name='carbonate',
                                               base_species=Species(name='CO2aq'),
                                               constraint=<Constraint.TOTAL: 0>,
                                               charge=0),

## Calculation

In [7]:
f_gas = eq_solver.FitFunc(system=s_gas, cond=c_gas)
r_gas = f_gas.solve()
f_nogas = eq_solver.FitFunc(system=s_nogas, cond=c_nogas)
r_nogas = f_nogas.solve()
print(r_gas.rmse, r_nogas.rmse)

3.510833468576701e-16 3.8459253727671276e-16


In [9]:
print(f'pH before contact with gas: {r_nogas.pH():.2f}')
print(f'pH after contact with 400 ppm CO2: {r_gas.pH():.2f}')

pH before contact with gas: 8.22
pH after contact with 400 ppm CO2: 9.06


In [12]:
# comparison of carbonate species distribution
print('------ concentrations (aq.) ------')
df = pd.DataFrame([r_nogas.distribution('carbonate'), r_gas.distribution('carbonate')], index=['no gas', 'with gas'])
# mole of CO2(g) is meaningless because we can't determine the amount (volume) of gas phase
df

------ concentrations (aq.) ------


,Species(name='CO2aq'),Species(name='HCO3-'),Species(name='CO3^2-'),Species(name='CO2(g)')
no gas,0.000106,0.00979,0.000104,NaN
with gas,0.000014,0.00870,0.000644,0.0
